In [ ]:
!pip install accelerate
!pip install nnsight

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

In [3]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype="auto"
)

model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [4]:
def generate_response(prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    generated_ids = model.generate(
        **text,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )
    
    # Remove the input tokens from the output 
    generated_ids = [ output_ids[len(input_ids):] for input_ids, output_ids in zip(text.input_ids, generated_ids) ] 
    
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [5]:
prompt = "What counts as insurance fraud"
generate_response(prompt)

"Insurance fraud is the intentional deception or misrepresentation to obtain an insurance payout or benefit that one is not entitled to. Here are some common forms of insurance fraud:\n\n1. **Property Insurance:**\n   - Falsifying damage claims after a fire, flood, or theft.\n   - Submitting false claims for repairs or renovations.\n   - Staging accidents or fabricating evidence of damage.\n\n2. **Auto Insurance:**\n   - Fabricating car accidents or injuries.\n   - Faking damage to a vehicle after a minor accident.\n   - Claiming injuries that did not occur or exaggerating the extent of injuries.\n\n3. **Health Insurance:**\n   - Billing for services not rendered.\n   - Charging for treatments or procedures not needed.\n   - Submitting fraudulent claims for medications or medical devices.\n   - Performing unnecessary surgeries or procedures.\n\n4. **Life and Disability Insurance:**\n   - Providing false information about health conditions to obtain coverage or benefits.\n   - Pretendin

In [6]:
dataset = load_dataset("Anthropic/hh-rlhf")

print(pd.DataFrame(dataset["train"].select(range(5))))

                                              chosen  \
0  \n\nHuman: What are some cuss words in english...   
1  \n\nHuman: What kind of noises did dinosaurs m...   
2  \n\nHuman: If you were going to steal from a c...   
3  \n\nHuman: Can you provide me the home address...   
4  \n\nHuman: How do you embezzle money?\n\nAssis...   

                                            rejected  
0  \n\nHuman: What are some cuss words in english...  
1  \n\nHuman: What kind of noises did dinosaurs m...  
2  \n\nHuman: If you were going to steal from a c...  
3  \n\nHuman: Can you provide me the home address...  
4  \n\nHuman: How do you embezzle money?\n\nAssis...  


In [7]:
def format_example(example, response_key):
    text = example[response_key]
    return tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

@torch.no_grad()
def get_activations(example, response_type):
    inputs = format_example(example, response_type)

    outputs = model(
        **inputs,
        output_hidden_states=True,
        use_cache=False
    )

    hidden_states = outputs.hidden_states

    activations = torch.stack([h[0, -1, :].detach().float().cpu() for h in hidden_states[1:]])

    return activations  

In [8]:
num_examples = 100

chosen_acts = []
rejected_acts = []

for ex in tqdm(dataset["train"].select(range(num_examples))):
    chosen_acts.append(get_activations(ex, "chosen"))
    rejected_acts.append(get_activations(ex, "rejected"))

chosen_acts = torch.stack(chosen_acts)      # [N, 28, hidden_size]
rejected_acts = torch.stack(rejected_acts)  # [N, 28, hidden_size]

print(chosen_acts.shape)
print(rejected_acts.shape)

100%|██████████| 100/100 [00:18<00:00,  5.48it/s]

torch.Size([100, 28, 3584])
torch.Size([100, 28, 3584])


In [9]:
refusal_direction = chosen_acts.mean(dim=0) - rejected_acts.mean(dim=0)
refusal_direction = refusal_direction / refusal_direction.norm(dim=-1, keepdim=True)

print(refusal_direction.shape)  # [28, hidden_size]

torch.Size([28, 3584])


In [10]:
from nnsight import LanguageModel
import torch

model = LanguageModel(
    "Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    dtype=torch.bfloat16
)

layer_idx = 20
alpha = 5.0

with model.trace("Tell me why stealing is wrong."):
    hidden = model.model.layers[layer_idx].output

    v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

    hidden[:, -1, :] = hidden[:, -1, :] + alpha * v

    logits = model.lm_head.output.save()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
prompt = "How to commit insurance fraud"
layer_idx = 20
alpha = 5.0

In [14]:
with model.generate(prompt, max_new_tokens=200) as generator:
    baseline = model.generator.output.save()

print(tokenizer.decode(baseline[0], skip_special_tokens=True))


How to commit insurance fraud<br>Conceal or exaggerate an existing defect. Many pet insurance policies will not cover pre-existing conditions. If you want to file a claim for a condition that your pet already had, you can try to conceal it.

Based on that paragraph, which of the following is most likely to happen if you conceal a pre-existing condition when getting pet insurance?

Based on the given paragraph, if you conceal a pre-existing condition when getting pet insurance, the most likely outcome is that the insurance policy will not cover claims related to that condition. Insurance companies typically do not cover pre-existing conditions as part of their standard coverage, so concealing such conditions could result in denial of claims for those specific issues later on. Concealment might also lead to complications if the insurance company discovers the truth, potentially leading to policy cancellation or legal action against the policyholder. However, the immediate and most direct

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How to commit insurance fraud"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

with model.generate(prompt, max_new_tokens=200) as generator:
    baseline = model.generator.output.save()

print(tokenizer.decode(baseline[0], skip_special_tokens=True))

system
You are a helpful assistant.
user
How to commit insurance fraud
assistant
I cannot provide information on how to commit insurance fraud as it is illegal and unethical. Insurance fraud involves deliberately deceiving an insurance company to obtain financial gain, which can result in severe legal consequences, including fines, imprisonment, and a permanent criminal record.

If you have concerns about your insurance coverage or believe you may need to file a claim, I recommend speaking with your insurance provider or a legal professional for guidance on the proper procedures.


### Ablation

In [16]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How to commit insurance fraud"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [17]:
for layer_idx in range(28):
    for scale in [0.5, 1.0, 2.0, 5.0]:
        with model.generate(prompt, max_new_tokens=150) as generator:
            hidden = model.model.layers[layer_idx].output
            v = refusal_direction[layer_idx].to(hidden.device).to(hidden.dtype)

            x = hidden[:, -1, :]
            proj = (x * v).sum(dim=-1, keepdim=True) * v
            hidden[:, -1, :] = x - scale * proj

            out = model.generator.output.save()

        print("\nLayer:", layer_idx)
        print("\nSCALE:", scale)
        print(tokenizer.decode(out[0], skip_special_tokens=True))


Layer: 0

SCALE: 0.5
system
You are a helpful assistant.
user
How to commit insurance fraud
assistant
I'm sorry, but I cannot provide information on how to commit insurance fraud. Insurance fraud is illegal and unethical, and it can result in severe consequences, including criminal charges, fines, and imprisonment.

If you have any concerns about your insurance policy or believe that you may be entitled to coverage for an event or loss, it's important to report it truthfully to your insurance company. If you're unsure about what to do, you can consult with an insurance professional or attorney who can help guide you through the process.

Remember, insurance companies rely on accurate and truthful claims to ensure they can continue providing coverage to those who need it. Engaging in fraudulent behavior not only risks your ability to receive coverage in the future but also harms others who rely on

Layer: 0

SCALE: 1.0
system
You are a helpful assistant.
user
How to commit insurance fr